In [0]:
pip install librosa

In [0]:
import os
import librosa
import numpy as np
import pandas as pd
import tempfile
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.sql.functions import col, pandas_udf, current_timestamp, regexp_extract, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, ArrayType

os.environ['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'
if not os.path.exists('/tmp/numba_cache'):
    os.makedirs('/tmp/numba_cache')

# Configuration
dbutils.widgets.text("catalog_name", "fma_catalog", "Unity Catalog Name")
dbutils.widgets.dropdown("use_pca", "true", ["true", "false"], "Apply PCA Dimensionality Reduction")
catalog = dbutils.widgets.get("catalog_name")
use_pca = dbutils.widgets.get("use_pca") == "true"
bronze_table = f"{catalog}.bronze.audio_raw"
silver_table = f"{catalog}.silver.audio_unsupervised"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.silver")

print(f"Configuration: PCA={'Enabled' if use_pca else 'Disabled'}")

In [0]:
from pyspark.sql.functions import col, length
health_check_df = spark.table(f"{catalog}.bronze.audio_raw").select(
    col("track_id"),
    col("length"),
    col("path"),
    col("content").isNull().alias("is_content_null")
)

print("Data Health Summary:")
print(f" - Total Row Count: {health_check_df.count()}")
print(f" - Rows with Null ID: {health_check_df.filter(col('track_id').isNull()).count()}")
print(f" - Rows with Null Content: {health_check_df.filter(col('is_content_null') == True).count()}")
print(f" - Rows with 0-byte Length: {health_check_df.filter(col('length') == 0).count()}")

In [0]:
audio_schema = StructType([
    StructField("tempo", FloatType(), True),
    StructField("mfcc_means", ArrayType(FloatType()), True),
    StructField("mfcc_std", ArrayType(FloatType()), True),
    StructField("spectral_centroid_mean", FloatType(), True),
    StructField("spectral_centroid_std", FloatType(), True),
    StructField("chroma_means", ArrayType(FloatType()), True),
    StructField("chroma_std", ArrayType(FloatType()), True),
    StructField("spectral_rolloff_mean", FloatType(), True),
    StructField("spectral_rolloff_std", FloatType(), True),
    StructField("spectral_bandwidth_mean", FloatType(), True),
    StructField("spectral_bandwidth_std", FloatType(), True),
    StructField("spectral_contrast_means", ArrayType(FloatType()), True),
    StructField("spectral_contrast_std", ArrayType(FloatType()), True),
    StructField("zcr_mean", FloatType(), True),
    StructField("zcr_std", FloatType(), True),
    StructField("beat_strength", FloatType(), True)
])

@pandas_udf(audio_schema)
def process_audio_robust(content_series: pd.Series) -> pd.DataFrame:
    import os
    os.environ['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'
    import librosa
    import numpy as np
    
    results = []
    for content in content_series:
        tmp_path = None
        try:
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
                tmp.write(content)
                tmp_path = tmp.name
            
            # Load 30s of audio for better genre representation
            y, sr = librosa.load(tmp_path, duration=30, sr=22050)
            
            # Feature extraction for genre-discriminative clustering
            
            # Tempo and rhythm features
            tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
            beat_strength = float(np.mean(librosa.util.sync(np.vstack([librosa.feature.rms(y=y)]), beats, aggregate=np.median)))
            
            # MFCC features (timbre) - mean and variance for dynamic range
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            mfcc_means = np.mean(mfccs, axis=1).tolist()
            mfcc_std = np.std(mfccs, axis=1).tolist()
            
            # Spectral centroid (brightness)
            sc = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
            sc_mean = float(np.mean(sc))
            sc_std = float(np.std(sc))
            
            # Chroma features (harmony/tonality) - critical for genre distinction
            chroma = librosa.feature.chroma_stft(y=y, sr=sr)
            chroma_means = np.mean(chroma, axis=1).tolist()
            chroma_std = np.std(chroma, axis=1).tolist()
            
            # Spectral rolloff (frequency content distribution)
            rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
            rolloff_mean = float(np.mean(rolloff))
            rolloff_std = float(np.std(rolloff))
            
            # Spectral bandwidth (timbre texture)
            bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
            bandwidth_mean = float(np.mean(bandwidth))
            bandwidth_std = float(np.std(bandwidth))
            
            # Spectral contrast (timbral texture, distinguishes genre well)
            contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=6)
            contrast_means = np.mean(contrast, axis=1).tolist()
            contrast_std = np.std(contrast, axis=1).tolist()
            
            # Zero crossing rate (percussiveness/noisiness)
            zcr = librosa.feature.zero_crossing_rate(y)[0]
            zcr_mean = float(np.mean(zcr))
            zcr_std = float(np.std(zcr))
            
            results.append({
                "tempo": float(tempo),
                "mfcc_means": mfcc_means,
                "mfcc_std": mfcc_std,
                "spectral_centroid_mean": sc_mean,
                "spectral_centroid_std": sc_std,
                "chroma_means": chroma_means,
                "chroma_std": chroma_std,
                "spectral_rolloff_mean": rolloff_mean,
                "spectral_rolloff_std": rolloff_std,
                "spectral_bandwidth_mean": bandwidth_mean,
                "spectral_bandwidth_std": bandwidth_std,
                "spectral_contrast_means": contrast_means,
                "spectral_contrast_std": contrast_std,
                "zcr_mean": zcr_mean,
                "zcr_std": zcr_std,
                "beat_strength": beat_strength
            })
        except Exception:
            results.append({
                "tempo": None, "mfcc_means": None, "mfcc_std": None,
                "spectral_centroid_mean": None, "spectral_centroid_std": None,
                "chroma_means": None, "chroma_std": None,
                "spectral_rolloff_mean": None, "spectral_rolloff_std": None,
                "spectral_bandwidth_mean": None, "spectral_bandwidth_std": None,
                "spectral_contrast_means": None, "spectral_contrast_std": None,
                "zcr_mean": None, "zcr_std": None, "beat_strength": None
            })
        finally:
            if tmp_path and os.path.exists(tmp_path):
                os.remove(tmp_path)
    return pd.DataFrame(results)

print(f"Enhanced Extraction Engine ready - extracting 74 genre-discriminative features.")

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA

# Load and Repartition for Parallel Performance
bronze_df = spark.table(f"{catalog}.bronze.audio_raw").repartition(32)

# Extract Audio Features
enriched_df = (bronze_df
    .withColumn("audio_dna", process_audio_robust(col("content")))
    .select("track_id", "audio_dna.*")
    .filter(col("tempo").isNotNull()))

# Flatten array features for the StandardScaler
# MFCCs: 13 means + 13 std = 26 features
for i in range(13):
    enriched_df = enriched_df.withColumn(f"mfcc_mean_{i}", col("mfcc_means")[i])
    enriched_df = enriched_df.withColumn(f"mfcc_std_{i}", col("mfcc_std")[i])

# Chroma: 12 means + 12 std = 24 features
for i in range(12):
    enriched_df = enriched_df.withColumn(f"chroma_mean_{i}", col("chroma_means")[i])
    enriched_df = enriched_df.withColumn(f"chroma_std_{i}", col("chroma_std")[i])

# Spectral Contrast: 7 means + 7 std = 14 features
for i in range(7):
    enriched_df = enriched_df.withColumn(f"contrast_mean_{i}", col("spectral_contrast_means")[i])
    enriched_df = enriched_df.withColumn(f"contrast_std_{i}", col("spectral_contrast_std")[i])

# Build complete feature list: 74 features total
# 1 tempo + 26 mfcc + 2 spectral_centroid + 24 chroma + 2 rolloff + 2 bandwidth + 14 contrast + 2 zcr + 1 beat_strength
feature_cols = (
    ["tempo", "beat_strength"] +
    [f"mfcc_mean_{i}" for i in range(13)] +
    [f"mfcc_std_{i}" for i in range(13)] +
    ["spectral_centroid_mean", "spectral_centroid_std"] +
    [f"chroma_mean_{i}" for i in range(12)] +
    [f"chroma_std_{i}" for i in range(12)] +
    ["spectral_rolloff_mean", "spectral_rolloff_std"] +
    ["spectral_bandwidth_mean", "spectral_bandwidth_std"] +
    [f"contrast_mean_{i}" for i in range(7)] +
    [f"contrast_std_{i}" for i in range(7)] +
    ["zcr_mean", "zcr_std"]
)

print(f"Assembling {len(feature_cols)} genre-discriminative features...")

# Apply Z-Score Normalization 
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features", handleInvalid="skip")
vectorized_df = assembler.transform(enriched_df)

scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", 
                        withStd=True, withMean=True)
scaler_model = scaler.fit(vectorized_df)
normalized_df = scaler_model.transform(vectorized_df)

# Optional: Apply PCA for dimensionality reduction
if use_pca:
    print("Applying PCA dimensionality reduction (preserving 95% variance)...")
    pca = PCA(k=None, inputCol="scaled_features", outputCol="pca_features")
    pca.setParams(varianceThreshold=0.95)  # Preserve 95% of variance
    pca_model = pca.fit(normalized_df)
    
    # Get number of components selected
    n_components = len(pca_model.explainedVariance)
    cumulative_variance = sum(pca_model.explainedVariance)
    
    print(f"PCA reduced 74 features to {n_components} components")
    print(f"Cumulative explained variance: {cumulative_variance:.4f}")
    
    normalized_df = pca_model.transform(normalized_df)
    features_col = "pca_features"
else:
    print("Using full 74-dimensional feature space (PCA disabled)")
    features_col = "scaled_features"

final_silver_df = (normalized_df
    .withColumn("processed_at", current_timestamp())
    .select("track_id", col(features_col).alias("scaled_features"), "processed_at"))

(final_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.audio_unsupervised"))

feature_desc = f"{n_components} PCA components" if use_pca else "74 features"
print(f"Silver Layer Complete: {final_silver_df.count()} tracks normalized with {feature_desc}.")